In [8]:
"""
Threads user search pipeline (EnsembleData) — schema-fixed version

What it does:
1) Reads CSV: Threads_accounts.csv with columns: brand, username
2) Calls /threads/user/search for each username
3) Saves raw JSON to: Threads/user_search/<username>.json
4) Loads all JSON files in that folder
5) Parses ALL returned candidates (each data[i]["node"]) into a table with columns:
   brand, query_username, candidate_rank,
   id/pk, username, full_name,
   is_verified, follower_count, text_post_app_is_private,
   profile_pic_url, scrape_ts
6) Saves parsed table to: threads_user_search_candidates.csv

Setup:
- pip install pandas requests
- export ENSEMBLEDATA_TOKEN="YOUR-TOKEN-HERE"
"""

from __future__ import annotations

import os
import re
import json
import time
import logging
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests

In [9]:
# -----------------------
# Config
# -----------------------
ROOT = "https://ensembledata.com/apis"
ENDPOINT = "/threads/user/search"
TOKEN = os.getenv("ENSEMBLEDATA_TOKEN", "GobJlP7vKjn4Z5Jw")

INPUT_CSV = Path("accounts.csv")  # columns: brand, username
OUT_DIR = Path("Threads") / "user_search"

SLEEP_SECS = 0.35          # small delay between calls
TIMEOUT_SECS = 60
OVERWRITE_EXISTING = False  # set True if you want to re-scrape even when JSON exists


# -----------------------
# Helpers
# -----------------------
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def safe_filename(name: str, max_len: int = 180) -> str:
    """
    Make a filename safe across OSes.
    """
    name = str(name).strip()
    name = re.sub(r"[^\w\-\.]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return (name or "unknown")[:max_len]

def extract_nodes(search_json: Any) -> List[Dict[str, Any]]:
    """
    EnsembleData Threads user search schema (per your sample):
      { "data": [ { "node": {...user...} }, { "node": {...} }, ... ] }

    Returns a list of user dicts (each is item["node"]).
    """
    if not isinstance(search_json, dict):
        return []

    data = search_json.get("data", [])
    if not isinstance(data, list):
        return []

    nodes: List[Dict[str, Any]] = []
    for item in data:
        if isinstance(item, dict) and isinstance(item.get("node"), dict):
            nodes.append(item["node"])
    return nodes

def to_int_or_none(x: Any) -> Optional[int]:
    try:
        if x is None:
            return None
        # sometimes pk comes as string
        return int(x)
    except Exception:
        return None

In [10]:
# -----------------------
# Step 1: Scrape and save raw JSONs
# -----------------------
def scrape_user_search(username: str, token: str) -> Dict[str, Any]:
    """
    Calls /threads/user/search?name=<username>&token=<token>
    Wraps raw JSON with _meta that includes scrape_ts.
    """
    url = ROOT + ENDPOINT
    params = {"name": username, "token": TOKEN}
    res = requests.get(url, params=params, timeout=TIMEOUT_SECS)
    res.raise_for_status()

    raw_json = res.json()
    return {
        "_meta": {
            "scrape_ts": utc_now_iso(),
            "endpoint": ENDPOINT,
            "query_name": username,
            "http_status": res.status_code,
            "request_url": res.url,
        },
        "data": raw_json,
    }

def run_scrape_from_csv() -> None:
    #token = os.environ.get("GobJlP7vKjn4Z5Jw")
    #if not token:
    #    raise RuntimeError("Missing ENSEMBLEDATA_TOKEN env var. Export it first.")

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(INPUT_CSV)
    required = {"brand", "username"}
    if not required.issubset(df.columns):
        raise ValueError(f"CSV must have columns {required}. Found: {list(df.columns)}")

    logging.info("Loaded %d rows from %s", len(df), INPUT_CSV)

    for i, row in df.iterrows():
        username = str(row["username"]).strip()
        if not username or username.lower() == "nan":
            continue

        out_path = OUT_DIR / f"{safe_filename(username)}.json"

        if out_path.exists() and not OVERWRITE_EXISTING:
            logging.info("[%d/%d] Exists, skipping: %s", i + 1, len(df), out_path.name)
            continue

        try:
            logging.info("[%d/%d] Scraping user/search for: %s", i + 1, len(df), username)
            wrapper = scrape_user_search(username=username, token=TOKEN)
            out_path.write_text(json.dumps(wrapper, ensure_ascii=False, indent=2), encoding="utf-8")
            time.sleep(SLEEP_SECS)

        except requests.HTTPError as e:
            err_wrapper = {
                "_meta": {
                    "scrape_ts": utc_now_iso(),
                    "endpoint": ENDPOINT,
                    "query_name": username,
                    "error": f"HTTPError: {e}",
                },
                "data": None,
            }
            out_path.write_text(json.dumps(err_wrapper, ensure_ascii=False, indent=2), encoding="utf-8")
            logging.exception("HTTP error for %s", username)

        except Exception as e:
            err_wrapper = {
                "_meta": {
                    "scrape_ts": utc_now_iso(),
                    "endpoint": ENDPOINT,
                    "query_name": username,
                    "error": f"Exception: {e}",
                },
                "data": None,
            }
            out_path.write_text(json.dumps(err_wrapper, ensure_ascii=False, indent=2), encoding="utf-8")
            logging.exception("Unexpected error for %s", username)


In [11]:
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

    # 1) Scrape user search results (raw JSON files)
    run_scrape_from_csv()

INFO: Loaded 2 rows from accounts.csv
INFO: [1/2] Scraping user/search for: adidas
INFO: [2/2] Scraping user/search for: nike


In [16]:
# -----------------------
# Step 2: Parse all JSONs into a candidate table
# -----------------------
def parse_all_jsons_to_table(
    out_csv: Path = Path("Threads/threads_user_search.csv")
) -> pd.DataFrame:
    if not OUT_DIR.exists():
        raise FileNotFoundError(f"Folder not found: {OUT_DIR}. Run scraping first.")

    df_in = pd.read_csv(INPUT_CSV)

    # Map: query_username -> brand (if duplicates exist, last one wins)
    brand_map = {
        str(u).strip(): str(b).strip()
        for b, u in zip(df_in["brand"], df_in["username"])
        if pd.notna(u)
    }

    rows: List[Dict[str, Any]] = []

    for fp in sorted(OUT_DIR.glob("*.json")):
        wrapper = json.loads(fp.read_text(encoding="utf-8"))

        meta = wrapper.get("_meta", {}) if isinstance(wrapper, dict) else {}
        query_username = meta.get("query_name") or fp.stem
        scrape_ts = meta.get("scrape_ts")

        raw_payload = wrapper.get("data") if isinstance(wrapper, dict) else None
        if not isinstance(raw_payload, dict):
            # still emit a row so you can see failures
            rows.append({
                "brand": brand_map.get(query_username),
                "query_username": query_username,
                "candidate_rank": None,
                "user_id": None,
                "username": None,
                "full_name": None,
                "is_verified": None,
                "follower_count": None,
                "text_post_app_is_private": None,
                "profile_pic_url": None,
                "scrape_ts": scrape_ts,
                "source_file": fp.name,
            })
            continue

        nodes = extract_nodes(raw_payload)

        if not nodes:
            rows.append({
                "brand": brand_map.get(query_username),
                "query_username": query_username,
                "candidate_rank": None,
                "user_id": None,
                "username": None,
                "full_name": None,
                "is_verified": None,
                "follower_count": None,
                "text_post_app_is_private": None,
                "profile_pic_url": None,
                "scrape_ts": scrape_ts,
                "source_file": fp.name,
            })
            continue

        for idx, node in enumerate(nodes, start=1):
            # prefer pk; fallback to id
            id_pk = node.get("pk", node.get("id"))
            id_pk_int = to_int_or_none(id_pk)

            follower_count = to_int_or_none(node.get("follower_count"))

            rows.append({
                "brand": brand_map.get(query_username),
                "query_username": query_username,
                "candidate_rank": idx,
                "user_id": id_pk_int if id_pk_int is not None else id_pk,
                "username": node.get("username"),
                "full_name": node.get("full_name"),
                "is_verified": node.get("is_verified"),
                "follower_count": follower_count,
                "text_post_app_is_private": node.get("text_post_app_is_private"),
                "profile_pic_url": node.get("profile_pic_url"),
                "scrape_ts": scrape_ts,
                "source_file": fp.name,
            })

    df_out = pd.DataFrame(rows)
    df_out.sort_values(["query_username", "candidate_rank"], inplace=True, na_position="last")
    df_out.to_csv(out_csv, index=False)
    logging.info("Saved parsed candidates: %s (%d rows)", out_csv, len(df_out))
    return df_out

In [17]:
# -----------------------
# Main
# -----------------------
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

    # 2) Parse all saved JSONs into a candidate table
    parse_all_jsons_to_table()


INFO: Saved parsed candidates: Threads/threads_user_search.csv (18 rows)
